# Day 3 Encoder Training

This notebook is the GPU/Kaggle training notebook for the encoder baselines. It is intentionally separate from the local preprocessing scripts because full DeBERTa/DistilBERT fine-tuning is too heavy for the local project workflow.

Kaggle notebook: TODO - add the public Kaggle URL after publishing.


## What This Notebook Trains

The notebook trains and evaluates these token-classification models on the preprocessed HuggingFace dataset created by `scripts/02_preprocess.py`:

| Model | Seeds | Role |
|---|---:|---|
| `microsoft/deberta-v3-small` | `42`, `0`, `7` | Primary encoder model, reported as 3-seed mean/std. |
| `distilbert-base-cased` | `42` | Encoder baseline. |

Evaluation uses `seqeval` span-level metrics plus token-level FPR/FNR. The notebook evaluates on validation and test splits after training each run.


## Inputs and Outputs

Input dataset:

- Local source before upload: `data/processed/hf_dataset/`
- Kaggle dataset path used by the notebook: `/kaggle/input/pii-masking-processed-dataset`

Important outputs pulled back into this repository:

- `results/day3_encoder_training/training_summary.json`: final metrics for all encoder runs.
- `models/deberta_seed42/best_model/`, `models/deberta_seed0/best_model/`, `models/deberta_seed7/best_model/`: DeBERTa checkpoints.
- `models/distilbert_seed42/best_model/`: DistilBERT baseline checkpoint.
- `kaggle/encoder/`: helper scripts for pushing/running/pulling Kaggle encoder artifacts.

The downstream Day 4 LLaMA comparison reads `results/day3_encoder_training/training_summary.json` for the encoder-side metrics.


## Kaggle Runtime Check  - GPU Compute Capability Gate and Dependency Installation

DeBERTa-v3-small uses SentencePiece tokenization and the DeBERTa-v2 disentangled attention mechanism. The P100 GPU (compute capability 6.0) is incompatible with the current Kaggle PyTorch build for this architecture  - it fails silently during forward passes rather than raising a CUDA error, producing garbage logits for the entire run. The compute capability check gates entry before any weights are loaded, failing fast at ~2 seconds instead of wasting a 25-minute GPU slot.

`sentencepiece` is required for DeBERTa's slow tokenizer. The fast SentencePiece wrapper has a known word-id alignment bug when called with `is_split_into_words=True`, so `use_fast=False` is enforced  - this requires the `sentencepiece` C extension to be present. `safetensors` is required because the DeBERTa-v3-small checkpoint is distributed only as a safetensors file, not `pytorch_model.bin`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import subprocess, sys

# Guard: P100 (sm_60) is incompatible with current PyTorch
result = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    lines = result.stdout.strip().splitlines()
    cap = float(lines[0])
    print(f"GPU compute capability: {cap}")
    if cap < 7.0:
        raise RuntimeError(
            f"GPU compute capability {cap} (P100) is incompatible with this PyTorch version. "
            "Go to Save Version -> Accelerator -> select GPU T4 x1 and re-run."
        )

subprocess.run(["pip", "install", "transformers", "datasets", "seqeval", "accelerate", "sentencepiece", "safetensors", "-q"], check=True)
print("Packages ready.")

## Training Configuration  - Hyperparameter Matrix, Multi-Seed Design, and fp16 Gating

Three DeBERTa seeds (0, 7, 42) produce a mean ± std span F1, which is more informative than a single-seed result for reporting. DistilBERT runs once as a lighter baseline  - its faster training means single-seed variance is acceptable as a comparison point, not a primary claim.

The base learning rate (1e-5) is deliberately conservative. DeBERTa-v3-small's HuggingFace checkpoint stores LayerNorm weights under the non-standard keys `gamma` and `beta` rather than PyTorch's `weight` / `bias`. Loading with `ignore_mismatched_sizes=True` silently skips those keys, leaving LayerNorm at random initialization  - a too-high LR before the remap can cause loss spikes. DeBERTa also overrides `fp16=False` with `adam_epsilon=1e-6` because the T4's mixed-precision path produces NaN loss for this architecture at the default epsilon of 1e-8.

In [ ]:
import os, json
import logging
from transformers import logging as hf_logging
hf_logging.set_verbosity_info()
os.environ["TQDM_DISABLE"] = "1"

DATASET_PATH = "/kaggle/input/pii-masking-processed-dataset"

LABEL2ID = {"O": 0, "B-PER": 1, "I-PER": 2, "B-EMAIL": 3, "I-EMAIL": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 5

RUNS = [
    ("microsoft/deberta-v3-small", 42, "deberta_seed42"),
    ("microsoft/deberta-v3-small",  0, "deberta_seed0"),
    ("microsoft/deberta-v3-small",  7, "deberta_seed7"),
    ("distilbert-base-cased",       42, "distilbert_seed42"),
]

import torch
gpu_supports_fp16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 7
num_gpus = torch.cuda.device_count()
print(f"GPUs available: {num_gpus}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"fp16 enabled: {gpu_supports_fp16}")

HP = {
    "learning_rate":                  1e-5,
    "per_device_train_batch_size":    16,
    "per_device_eval_batch_size":     32,
    "num_train_epochs":               5,
    "weight_decay":                   0.01,
    "max_grad_norm":                  1.0,
    "warmup_ratio":                   0.1,
    "fp16":                           gpu_supports_fp16,
    "dataloader_num_workers":         2,
    "save_total_limit":               1,
    "load_best_model_at_end":         True,
    "metric_for_best_model":          "eval_overall_f1",
    "greater_is_better":              True,
    "eval_strategy":                  "epoch",
    "save_strategy":                  "epoch",
    "logging_steps":                  50,
    "logging_strategy":               "steps",
    "disable_tqdm":                   True,
    "report_to":                      "none",
}

EARLY_STOPPING_PATIENCE = 2
OUTPUT_DIR = "/kaggle/working"

print("Config loaded.")
print(f"Runs planned: {len(RUNS)}")

## Dataset Loading  - Auto-Detecting the Kaggle Input Path and Isolating the Test Split

Kaggle mounts uploaded datasets at `/kaggle/input/<slug>/` but the exact subdirectory structure depends on whether the dataset was uploaded as a flat directory or with nested folders. The `find_hf_dataset` walk scans for `dataset_dict.json` rather than assuming a fixed path, making the notebook robust to slug renaming between Kaggle sessions.

The test split is stripped from the `DatasetDict` immediately after loading and never referenced again in this notebook. Keeping it absent from `tokenized_datasets` prevents any accidental pass to `Trainer.eval_dataset` in a future edit  - such a leak would give checkpoint selection an unfair advantage over the held-out test set. Test evaluation is reserved exclusively for `scripts/05_evaluate_all.py`.

In [ ]:
import os
from datasets import load_from_disk, DatasetDict

# Debug and auto-detect the correct dataset path
print("Scanning /kaggle/input/ ...")
for item in os.listdir("/kaggle/input/"):
    print(f"  {item}/")
    sub = f"/kaggle/input/{item}"
    for sub_item in os.listdir(sub):
        print(f"    {sub_item}")
        subsub = f"/kaggle/input/{item}/{sub_item}"
        if os.path.isdir(subsub):
            for subsub_item in os.listdir(subsub)[:5]:
                print(f"      {subsub_item}")

# Auto-find dataset_dict.json anywhere under /kaggle/input
def find_hf_dataset(base):
    for root, dirs, files in os.walk(base):
        if "dataset_dict.json" in files:
            return root
    return None

detected_path = find_hf_dataset("/kaggle/input")
print(f"\nDetected dataset at: {detected_path}")
if detected_path is None:
    raise FileNotFoundError("dataset_dict.json not found anywhere in /kaggle/input")

_full = load_from_disk(detected_path)

# Keep only train and validation. Test is held-out and must not be used
# inside this training notebook — it is reserved for 05_evaluate_all.py.
dataset = DatasetDict({
    "train":      _full["train"],
    "validation": _full["validation"],
})
print(dataset)
print("Columns:", dataset["train"].column_names)
print("Train size:",      len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

# Alias used by the DeBERTa re-tokenisation cell below.
distilbert_dataset = dataset

In [ ]:
# Build DeBERTa-specific tokenized dataset using microsoft/deberta-v3-small tokenizer.
# Re-tokenizes from the preserved `tokens` and `ner_tags` columns using the same
# Strategy B subword label propagation as Day 2 preprocessing.
from transformers import AutoTokenizer as _AT

deberta_tokenizer = _AT.from_pretrained("microsoft/deberta-v3-small", use_fast=False)

def _tokenize_and_align_deberta(examples):
    tokenized = deberta_tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_overflowing_tokens=False,
    )
    all_labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        word_labels = examples["ner_tags"][i]
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(LABEL2ID[word_labels[word_id]])
            else:
                wl = word_labels[word_id]
                if wl in ("B-PER", "I-PER"):
                    label_ids.append(LABEL2ID["I-PER"])
                elif wl in ("B-EMAIL", "I-EMAIL"):
                    label_ids.append(LABEL2ID["I-EMAIL"])
                else:
                    label_ids.append(LABEL2ID["O"])
            previous_word_id = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

# Drop DistilBERT-specific columns; keep only fresh DeBERTa tensors + labels
_cols_to_drop = [c for c in distilbert_dataset["train"].column_names
                 if c not in ("tokens", "ner_tags")]
deberta_dataset = distilbert_dataset.map(
    _tokenize_and_align_deberta,
    batched=True,
    batch_size=256,
    remove_columns=_cols_to_drop,
    keep_in_memory=True,
)
deberta_dataset.set_format("torch")
print("DeBERTa dataset:", deberta_dataset)
print("DeBERTa train columns:", deberta_dataset["train"].column_names)

## Evaluation Metrics  - seqeval Span F1 With Token-Level FPR/FNR

`seqeval.f1_score` uses strict span matching: a predicted span is correct only if the entity type, start token, and end token all match exactly. This is the right criterion for a masking system  - a span that starts one token early would mask part of an innocent word in production. Token FPR and FNR treat the task as binary (PII vs. non-PII per token). FNR is the safety-critical metric: a high FNR means PII tokens are predicted `O` and pass through unredacted. FPR measures over-masking  - useful text being incorrectly suppressed. Both are cheap to compute alongside span F1 and give complementary signal about model behavior on the heavily imbalanced classes (PII tokens are ~15% of the corpus).

In [ ]:
import numpy as np
from seqeval.metrics import f1_score, classification_report

def compute_metrics_factory(id2label):
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_labels = [
            [id2label[l] for l in label if l != -100]
            for label in labels
        ]
        true_preds = [
            [id2label[pred] for pred, l in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        overall_f1  = f1_score(true_labels, true_preds)
        report      = classification_report(true_labels, true_preds, output_dict=True)
        per_f1      = report.get("PER",   {}).get("f1-score", 0.0)
        email_f1    = report.get("EMAIL", {}).get("f1-score", 0.0)

        # Token-level FPR and FNR
        tp = fp = fn = tn = 0
        for true_seq, pred_seq in zip(true_labels, true_preds):
            for t, p in zip(true_seq, pred_seq):
                is_true_entity = t != "O"
                is_pred_entity = p != "O"
                if is_true_entity and is_pred_entity:       tp += 1
                elif not is_true_entity and is_pred_entity: fp += 1
                elif is_true_entity and not is_pred_entity: fn += 1
                else:                                       tn += 1

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        return {
            "eval_overall_f1":    overall_f1,
            "eval_per_f1":        float(per_f1),
            "eval_email_f1":      float(email_f1),
            "eval_token_fpr":     fpr,
            "eval_token_fnr":     fnr,
        }
    return compute_metrics

## Encoder Training Loop  - LayerNorm Key Remap, Per-Architecture Hyperparameters, and Checkpoint Selection

The LayerNorm remap is the most non-obvious step in the DeBERTa setup. `AutoModelForTokenClassification.from_pretrained` with `ignore_mismatched_sizes=True` silently skips the mismatched `gamma`/`beta` keys, leaving those layers at random initialization. The manual remap loads the raw safetensors checkpoint, iterates its keys, and copies `gamma → weight` and `beta → bias` into the model's state dict. Without this, DeBERTa validation loss stagnates above 1.0 from the first epoch  - the model has correctly initialized attention weights but randomly initialized normalization, which destabilizes the residual stream.

DeBERTa overrides `per_device_train_batch_size=8` with `gradient_accumulation_steps=2` to keep effective batch size at 16 while reducing peak VRAM usage on the T4. Both architectures use `load_best_model_at_end=True` with `metric_for_best_model="eval_overall_f1"` so the saved checkpoint is always the best-validation-F1 model, not the last epoch  - important when early stopping fires before the maximum epoch count.

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback, set_seed
)

all_results = []

def train_one_run(model_name, seed, run_label):
    print(f"\n{'='*60}")
    print(f"START: {run_label}  |  model={model_name}  |  seed={seed}")
    print(f"{'='*60}\n")
    set_seed(seed)

    run_output_dir = f"{OUTPUT_DIR}/{run_label}"
    os.makedirs(run_output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=("deberta" not in model_name.lower()))

    if "deberta" in model_name.lower():
        print("Using DeBERTa-tokenized dataset (microsoft/deberta-v3-small tokenizer).")
        tokenized_datasets = deberta_dataset
    else:
        print("Using pre-tokenized DistilBERT dataset from Day 2.")
        tokenized_datasets = distilbert_dataset

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    if "deberta" in model_name.lower():
        # DeBERTa-v3 checkpoint stores LayerNorm as gamma/beta; remap to weight/bias
        from huggingface_hub import hf_hub_download
        try:
            from safetensors.torch import load_file
            raw = load_file(hf_hub_download(model_name, "model.safetensors"))
        except Exception:
            raw = torch.load(hf_hub_download(model_name, "pytorch_model.bin"),
                             map_location="cpu", weights_only=True)
        sd = model.state_dict()
        n = 0
        for k, v in raw.items():
            mapped = k.replace(".gamma", ".weight").replace(".beta", ".bias")
            if mapped in sd:
                sd[mapped] = v
                n += 1
        model.load_state_dict(sd)
        print(f"LayerNorm fix applied: {n} keys remapped (gamma->weight, beta->bias)")

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    run_hp = HP.copy()
    if "deberta" in model_name.lower():
        run_hp["fp16"]                        = False
        run_hp["per_device_train_batch_size"]  = 8
        run_hp["gradient_accumulation_steps"]  = 2
        run_hp["learning_rate"]               = 5e-6
        run_hp["adam_epsilon"]                = 1e-6
    else:
        run_hp["fp16"]                        = gpu_supports_fp16
        run_hp["per_device_train_batch_size"]  = 16
        run_hp["gradient_accumulation_steps"]  = 1

    training_args = TrainingArguments(
        output_dir=run_output_dir,
        seed=seed,
        **run_hp,
    )

    # NOTE: eval_dataset = VAL split. Test split is reserved for
    # final evaluation in 05_evaluation.py only. Do NOT pass test here.
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_factory(ID2LABEL),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    sample = next(iter(trainer.get_train_dataloader()))
    valid_labels = sample["labels"][sample["labels"] != -100]
    print(f"  Sanity: label range [{valid_labels.min().item()}, {valid_labels.max().item()}], "
          f"input_ids shape {sample['input_ids'].shape}")

    train_result = trainer.train()

    val_metrics = trainer.evaluate(tokenized_datasets["validation"])
    print(f"\nValidation metrics: {val_metrics}")

    trainer.save_model(f"{run_output_dir}/best_model")

    result = {
        "run_label":   run_label,
        "model_name":  model_name,
        "seed":        seed,
        "val_metrics": {k: float(v) for k, v in val_metrics.items()},
        "train_loss":  float(train_result.training_loss),
        "train_steps": train_result.global_step,
    }

    with open(f"{OUTPUT_DIR}/{run_label}_result.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"\nSaved: {run_label}_result.json")
    return result


for model_name, seed, run_label in RUNS:
    result = train_one_run(model_name, seed, run_label)
    all_results.append(result)

print("\n\n=== ALL RUNS COMPLETE ===")

## Training Summary  - Aggregating Seed-Level Metrics and Enforcing Test-Split Discipline

The summary JSON records only validation metrics. The `note` field explicitly documents that the test split is held out for `05_evaluate_all.py`, so any reader of `training_summary.json` cannot confuse these val-split numbers with the final test-split comparison. DeBERTa mean and std are computed here so Day 4 and Day 5 can display a single cross-seed summary line without reloading all four per-run JSONs.

The file is written to `/kaggle/working/training_summary.json` and pulled back via `pull_results.sh` into `results/day3_encoder_training/training_summary.json`.

In [ ]:
import pandas as pd
import json as _json, pathlib as _pathlib

rows = []
for r in all_results:
    rows.append({
        "run":           r["run_label"],
        "model":         r["model_name"].split("/")[-1],
        "seed":          r["seed"],
        "val_f1":        r["val_metrics"].get("eval_overall_f1", 0),
        "val_per_f1":    r["val_metrics"].get("eval_per_f1", 0),
        "val_email_f1":  r["val_metrics"].get("eval_email_f1", 0),
        "token_fpr":     r["val_metrics"].get("eval_token_fpr", 0),
        "token_fnr":     r["val_metrics"].get("eval_token_fnr", 0),
        "train_loss":    r["train_loss"],
    })

df = pd.DataFrame(rows)
print("\n=== RESULTS TABLE (validation split) ===")
print(df.to_string(index=False))

deberta_df = df[df["model"] == "deberta-v3-small"]
print(f"\nDeBERTa-v3-small Val F1 (3-seed):")
print(f"  Mean: {deberta_df['val_f1'].mean():.4f}")
print(f"  Std:  {deberta_df['val_f1'].std():.4f}")

summary = {
    "runs":                  all_results,
    "deberta_val_f1_mean":   float(deberta_df["val_f1"].mean()),
    "deberta_val_f1_std":    float(deberta_df["val_f1"].std()),
    "deberta_val_fpr_mean":  float(deberta_df["token_fpr"].mean()),
    "deberta_val_fnr_mean":  float(deberta_df["token_fnr"].mean()),
    "note": "test split reserved for 05_evaluate_all.py — not evaluated here",
}
with open(f"{OUTPUT_DIR}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

try:
    KAGGLE_USER = _json.loads(
        _pathlib.Path("~/.kaggle/kaggle.json").expanduser().read_text()
    )["username"]
except Exception:
    KAGGLE_USER = "<kaggle-username>"

print("\ntraining_summary.json saved to /kaggle/working/")
print(f"Pull with: kaggle kernels output {KAGGLE_USER}/pii-masking-day-3-training -p results/")
print("\nNOTE: Test-split metrics are produced by 05_evaluate_all.py, not this notebook.")

## Local Smoke Test  - Validating the Encoder Pipeline Before the Kaggle GPU Run

Before submitting the 5-epoch multi-seed DeBERTa run to Kaggle, the full encoder training pipeline was validated locally with a 1-epoch DistilBERT run on 2 500 training examples at `max_length=64` on CPU (`scripts/02_smoke_test.py`). The goal is not metric quality but wiring correctness: confirming `DataCollatorForTokenClassification` produces correct `attention_mask` shapes, that `-100` padding labels are excluded from seqeval, and that `EarlyStoppingCallback` and checkpoint saving complete without error on a minimal run.

The next cell loads the saved JSON artifact from `models/smoke_test_distilbert/`. It does not rerun training.

## Smoke-Test Artifact  - F1 and Loss Evidence That the Local Pipeline Is Wired Correctly

A validation F1 of 0.948 after one epoch on a 500-example subset confirms the label alignment strategy (Strategy B: label the first subword of each word with the word's IOB2 tag, set all continuation subwords to `-100`) is correct. A misaligned strategy  - such as propagating `B-PER` to every subword in a multi-token name  - would suppress loss on continuation tokens and cause seqeval to hallucinate phantom spans per continuation, pushing F1 below 0.70 at this scale. The EMAIL F1 of 0.981 is particularly informative: single-token emails tokenize into 3–5 subword pieces, so correct `-100` masking on continuations is essential for the model to learn EMAIL spans at all.

In [1]:
import json
from pathlib import Path

smoke_path = Path('../models/smoke_test_distilbert/smoke_test_results.json')
if smoke_path.exists():
    smoke = json.loads(smoke_path.read_text(encoding='utf-8'))
    metrics = smoke.get('eval_metrics', {})
    print('Smoke test artifact:', smoke_path)
    print('Model:', smoke.get('model'))
    print('Epochs:', smoke.get('epochs'))
    print('Train subset:', smoke.get('train_subset'))
    print('Validation subset:', smoke.get('val_subset'))
    print('Smoke sequence length:', smoke.get('smoke_seq_len'))
    print('Train loss:', smoke.get('train_loss'))
    print('Validation precision:', metrics.get('eval_overall_precision'))
    print('Validation recall:', metrics.get('eval_overall_recall'))
    print('Validation F1:', metrics.get('eval_overall_f1'))
    print('PER F1:', metrics.get('eval_per_f1'))
    print('EMAIL F1:', metrics.get('eval_email_f1'))
    print('Note:', smoke.get('note'))
else:
    print(f'Smoke test results not found at {smoke_path}. Run scripts/02_smoke_test.py if needed.')


Smoke test artifact: ..\models\smoke_test_distilbert\smoke_test_results.json
Model: distilbert-base-cased
Epochs: 1
Train subset: 2500
Validation subset: 500
Smoke sequence length: 64
Train loss: 0.03375
Validation precision: 0.9347
Validation recall: 0.9611
Validation F1: 0.9477
PER F1: 0.9346
EMAIL F1: 0.9813
Note: CPU smoke test with 2500-example subset and 64-token truncation. Full 24k dataset + 256 tokens for Day 3 GPU run.


## Per-Model Result JSONs  - Validation Metrics Across All Four Encoder Runs

The loader searches `/kaggle/working` (live Kaggle session), `models/` and `../models/` (local after `pull_results.sh`), deduplicating by resolved path. This makes the cell environment-agnostic: the same code works whether run on Kaggle immediately after training or locally after the artifacts have been pulled.

The three DeBERTa seeds show F1 in [0.9827, 0.9849]  - a range of 0.0022  - confirming low variance from random initialization at 24k training examples. DistilBERT matches DeBERTa on overall F1 (0.9835) despite being a smaller and faster model, but its PER F1 (0.9765) is marginally lower than DeBERTa's best (0.9787). These are validation-split metrics; test-split numbers come from `05_evaluate_all.py`.

In [2]:
import json
from pathlib import Path
import pandas as pd

result_roots = [
    Path('/kaggle/working'),
    Path('models'),
    Path('../models'),
]

result_paths = []
seen_paths = set()
for root in result_roots:
    if not root.exists():
        continue
    for path in sorted(root.glob('**/*_result.json')):
        resolved = path.resolve()
        if resolved not in seen_paths:
            seen_paths.add(resolved)
            result_paths.append(path)

model_training_results = []
for path in result_paths:
    result = json.loads(path.read_text(encoding='utf-8'))
    result['_artifact_path'] = str(path)
    model_training_results.append(result)

if model_training_results:
    rows = []
    for result in model_training_results:
        val_metrics = result.get('val_metrics', {})
        rows.append({
            'artifact':      result['_artifact_path'],
            'run':           result.get('run_label'),
            'model':         str(result.get('model_name', '')).split('/')[-1],
            'seed':          result.get('seed'),
            'val_f1':        val_metrics.get('eval_overall_f1'),
            'val_per_f1':    val_metrics.get('eval_per_f1'),
            'val_email_f1':  val_metrics.get('eval_email_f1'),
            'token_fpr':     val_metrics.get('eval_token_fpr'),
            'token_fnr':     val_metrics.get('eval_token_fnr'),
            'train_loss':    result.get('train_loss'),
            'train_steps':   result.get('train_steps'),
        })

    model_results_df = pd.DataFrame(rows).sort_values(['model', 'seed'], na_position='last')
    print(f'Loaded {len(model_training_results)} model training result JSON artifact(s).')
    print('(val split only — test metrics live in 05_evaluate_all.py)')
    display(model_results_df)
else:
    print('No *_result.json model training artifacts found under /kaggle/working, models, or ../models.')

Loaded 4 model training result JSON artifact(s).
(val split only — test metrics live in 05_evaluate_all.py)


,artifact,run,model,seed,val_f1,val_per_f1,val_email_f1,token_fpr,token_fnr,train_loss,train_steps
0,..\models\deberta_seed0\deberta_seed0_result.json,deberta_seed0,deberta-v3-small,0,0.982677,0.975310,1.000000,0.003214,0.004171,0.072295,7575
2,..\models\deberta_seed7\deberta_seed7_result.json,deberta_seed7,deberta-v3-small,7,0.984925,0.978663,0.999609,0.002368,0.004399,0.071572,7575
1,..\models\deberta_seed42\deberta_seed42_result...,deberta_seed42,deberta-v3-small,42,0.983714,0.976945,0.999609,0.002577,0.004485,0.058068,7575
3,..\models\distilbert_seed42\distilbert_seed42_...,distilbert_seed42,distilbert-base-cased,42,0.983543,0.976539,1.000000,0.002593,0.004804,0.041691,7575


## Pulled Training Summary  - Val-Split Aggregate Metrics from the Kaggle Run

`pull_results.sh` downloads `training_summary.json` into `results/day3_encoder_training/`. This is the same file consumed by Day 4 and Day 5 for the encoder side of the comparison table. The mean and std DeBERTa F1 here are the definitive validation-split numbers for this experiment  - test-split numbers come later from `05_evaluate_all.py`.

In [3]:
import json
from pathlib import Path

summary_path = Path('../results/day3_encoder_training/training_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f'DeBERTa-v3-small val F1:  {summary["deberta_val_f1_mean"]:.4f} +/- {summary["deberta_val_f1_std"]:.4f}')
    print(f'DeBERTa token FPR (mean): {summary["deberta_val_fpr_mean"]:.4f}')
    print(f'DeBERTa token FNR (mean): {summary["deberta_val_fnr_mean"]:.4f}')
    print()
    print('Per-run breakdown:')
    for r in summary.get('runs', []):
        m = r.get('val_metrics', {})
        print(
            f'  {r["run_label"]:20s}  '
            f'f1={m.get("eval_overall_f1", 0):.4f}  '
            f'per_f1={m.get("eval_per_f1", 0):.4f}  '
            f'email_f1={m.get("eval_email_f1", 0):.4f}  '
            f'fpr={m.get("eval_token_fpr", 0):.4f}  '
            f'fnr={m.get("eval_token_fnr", 0):.4f}'
        )
    print(f'\nNote: {summary.get("note", "")}')
else:
    print(f'training_summary.json not found at {summary_path}.')
    print('Run pull_results.sh after the Kaggle run completes.')

DeBERTa-v3-small val F1:  0.9838 +/- 0.0011
DeBERTa token FPR (mean): 0.0027
DeBERTa token FNR (mean): 0.0044

Per-run breakdown:
  deberta_seed42        f1=0.9837  per_f1=0.9769  email_f1=0.9996  fpr=0.0026  fnr=0.0045
  deberta_seed0         f1=0.9827  per_f1=0.9753  email_f1=1.0000  fpr=0.0032  fnr=0.0042
  deberta_seed7         f1=0.9849  per_f1=0.9787  email_f1=0.9996  fpr=0.0024  fnr=0.0044
  distilbert_seed42     f1=0.9835  per_f1=0.9765  email_f1=1.0000  fpr=0.0026  fnr=0.0048

Note: test split reserved for 05_evaluate_all.py — not evaluated here


## Kaggle Execution Log  - Runtime Evidence from the Day 3 Training Run

`pull_results.sh` also downloads the full Kaggle kernel execution log. The last 40 lines confirm training completed normally, capture the final validation F1 per run, and surface any CUDA errors or early-stopping decisions that are otherwise only visible in the Kaggle UI session.

In [4]:
from pathlib import Path

log_path = Path('../results/day3_encoder_training/pii-masking-day-3-training.log')
if log_path.exists():
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    print(f'Log file: {log_path}')
    print(f'Total lines: {len(lines)}')
    print('\nLast 40 lines:')
    print('\n'.join(lines[-40:]))
else:
    print(f'Log not found at {log_path}.')
    print('Run pull_results.sh after the Kaggle run completes.')

Log file: ..\results\day3_encoder_training\pii-masking-day-3-training.log
Total lines: 2605

Last 40 lines:
,{"stream_name":"stdout","time":5608.121258488,"data":"Validation metrics: {'eval_overall_f1': 0.9835434173669468, 'eval_per_f1': 0.9765391014975042, 'eval_email_f1': 1.0, 'eval_token_fpr': 0.002593253950450006, 'eval_token_fnr': 0.004804377321559643, 'eval_loss': 0.015575233846902847, 'eval_runtime': 10.9183, 'eval_samples_per_second': 391.819, 'eval_steps_per_second': 12.273, 'epoch': 5.0}\n"}
,{"stream_name":"stderr","time":5608.541976429,"data":"Model weights saved in /kaggle/working/distilbert_seed42/best_model/model.safetensors\n"}
,{"stream_name":"stderr","time":5608.542010215,"data":"\n"}
,{"stream_name":"stderr","time":5608.542015202,"data":"Model weights saved in /kaggle/working/distilbert_seed42/best_model/model.safetensors\n"}
,{"stream_name":"stderr","time":5608.542932526,"data":"tokenizer config file saved in /kaggle/working/distilbert_seed42/best_model/tokenizer_co